# Asset Embeddings: WRDS Replication and Extension

This notebook reconstructs the asset-embedding project using WRDS S12 and S34 holdings from 2005Q1 through 2024Q4. The workflow preserves raw extracts, creates reproducible intermediate datasets, and checkpoints computationally expensive models by quarter.

The notebook is designed to be executed sequentially. Every code cell is introduced by a Markdown cell explaining its purpose, inputs, outputs, and expected computational cost.

## 1. Imports and project configuration

This low-cost cell imports only packages needed for project setup, defines the analysis window, and creates canonical paths relative to the notebook. Keeping paths centralized prevents later cells from depending on the machine's current working directory.

In [7]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import sklearn

PROJECT_ROOT = Path.cwd().resolve()
START_QUARTER = pd.Period("2005Q1", freq="Q")
END_QUARTER = pd.Period("2024Q4", freq="Q")
QUARTERS = pd.period_range(START_QUARTER, END_QUARTER, freq="Q")

assert len(QUARTERS) == 80
print(f"Project root: {PROJECT_ROOT}")
print(f"Analysis window: {START_QUARTER} to {END_QUARTER} ({len(QUARTERS)} quarters)")

Project root: /Users/harmandeep/Desktop/Projects/Asset_Embeddings
Analysis window: 2005Q1 to 2024Q4 (80 quarters)


## 2. Create the reproducible project layout

This low-cost cell creates folders for immutable WRDS extracts, crosswalks, intermediate and processed data, model checkpoints, diagnostics, tables, figures, logs, reusable source code, and tests. It is idempotent and can be rerun safely.

In [8]:
DIRECTORIES = [
    "config",
    "src/wrds",
    "src/linking",
    "src/cleaning",
    "src/embeddings",
    "src/benchmarks",
    "src/reporting",
    "scripts",
    "data/raw/s12",
    "data/raw/s34",
    "data/raw/crsp",
    "data/raw/compustat",
    "data/crosswalks",
    "data/interim",
    "data/processed",
    "data/external",
    "artifacts/embeddings",
    "artifacts/models",
    "artifacts/diagnostics",
    "artifacts/checkpoints",
    "results/tables",
    "results/figures",
    "logs",
    "tests",
]

for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)

print(f"Verified {len(DIRECTORIES)} project directories.")

Verified 24 project directories.


## 3. Record the local execution environment

This low-cost cell writes a small JSON manifest containing the analysis window and package versions. It deliberately records only whether a WRDS username is configured, never the username or password themselves. The manifest helps diagnose reproducibility differences across machines and future runs.

In [9]:
environment_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "analysis_start": str(START_QUARTER),
    "analysis_end": str(END_QUARTER),
    "quarter_count": len(QUARTERS),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pa.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "wrds_username_configured": bool(os.getenv("WRDS_USERNAME")),
}

manifest_path = PROJECT_ROOT / "artifacts" / "diagnostics" / "environment.json"
manifest_path.write_text(json.dumps(environment_manifest, indent=2) + "\n")

display(pd.DataFrame({"value": environment_manifest["packages"]}).rename_axis("package"))
print(f"Environment manifest saved to: {manifest_path.relative_to(PROJECT_ROOT)}")
print(f"WRDS username configured: {environment_manifest['wrds_username_configured']}")

,value
package,
numpy,2.0.2
pandas,2.3.3
pyarrow,21.0.0
scikit_learn,1.6.1


Environment manifest saved to: artifacts/diagnostics/environment.json
WRDS username configured: False


## 4. Load the project-local WRDS client

Load the WRDS client from the project-local .packages directory and verify its version. This does not change the global Python environment.

In [ ]:
LOCAL_PACKAGES = PROJECT_ROOT / ".packages"
if not LOCAL_PACKAGES.exists():
    raise FileNotFoundError(f"Missing local package directory: {LOCAL_PACKAGES}")
if str(LOCAL_PACKAGES) not in sys.path:
    sys.path.insert(0, str(LOCAL_PACKAGES))
from importlib.metadata import version
import wrds
print(f"WRDS client version: {version('wrds')}")
print(f"WRDS client location: {Path(wrds.__file__).resolve()}")

## 5. Open an authenticated WRDS connection

Run this cell interactively. It asks for the WRDS username when the WRDS_USERNAME environment variable is absent. WRDS will securely prompt for a password if required; credentials are not saved in the project.

In [ ]:
WRDS_USERNAME = os.getenv("WRDS_USERNAME") or input("WRDS username: ").strip()
if not WRDS_USERNAME:
    raise ValueError("A WRDS username is required.")
db = wrds.Connection(wrds_username=WRDS_USERNAME)
print("WRDS connection established.")

## 6. Discover candidate S12 and S34 tables

List accessible WRDS libraries and inventory tables in holdings-related libraries. This is a low-cost metadata operation; it does not download holdings. Results are saved locally as Parquet.

In [ ]:
libraries = pd.DataFrame({"library": sorted(db.list_libraries())})
pattern = r"(^|_)(tfn|tr|refinitiv|thomson|s12|s34)(_|$)"
candidate_libraries = libraries.loc[
    libraries["library"].str.contains(pattern, case=False, regex=True), "library"
].tolist()
rows = [
    {"library": library, "table": table}
    for library in candidate_libraries
    for table in sorted(db.list_tables(library=library))
]
table_catalog = pd.DataFrame(rows, columns=["library", "table"])
table_catalog["is_s12_or_s34"] = table_catalog["table"].str.contains(
    r"s12|s34", case=False, regex=True
) if not table_catalog.empty else pd.Series(dtype=bool)
catalog_directory = PROJECT_ROOT / "data/crosswalks/wrds_catalog"
catalog_directory.mkdir(parents=True, exist_ok=True)
libraries.to_parquet(catalog_directory / "libraries.parquet", index=False)
table_catalog.to_parquet(catalog_directory / "candidate_tables.parquet", index=False)
print(f"Libraries: {len(libraries):,}; candidate tables: {len(table_catalog):,}")
display(table_catalog.sort_values(["is_s12_or_s34", "library", "table"], ascending=[False, True, True]).head(100))

## 7. Save S12 and S34 column metadata

Describe each candidate S12/S34 table and save its columns locally. Errors are recorded separately so one unavailable table does not stop discovery.

In [ ]:
schema_frames, schema_errors = [], []
targets = table_catalog.loc[table_catalog["is_s12_or_s34"], ["library", "table"]].drop_duplicates()
for target in targets.itertuples(index=False):
    try:
        frame = db.describe_table(library=target.library, table=target.table).reset_index()
        frame.insert(0, "table", target.table)
        frame.insert(0, "library", target.library)
        schema_frames.append(frame)
    except Exception as error:
        schema_errors.append({"library": target.library, "table": target.table, "error": str(error)})
column_catalog = pd.concat(schema_frames, ignore_index=True) if schema_frames else pd.DataFrame()
schema_error_table = pd.DataFrame(schema_errors)

# WRDS returns SQLAlchemy objects such as DATE() in metadata columns.
# Convert object-valued metadata to strings so PyArrow can serialize it.
for frame in (column_catalog, schema_error_table):
    for column in frame.select_dtypes(include=["object"]).columns:
        frame[column] = frame[column].map(lambda value: None if pd.isna(value) else str(value))
column_catalog.to_parquet(catalog_directory / "s12_s34_columns.parquet", index=False)
schema_error_table.to_parquet(catalog_directory / "schema_errors.parquet", index=False)
print(f"Described tables: {len(schema_frames):,}; errors: {len(schema_errors):,}")
display(column_catalog.head(200))

## 8. Select the authoritative WRDS source tables

The catalog exposes S12 in both the legacy tfn namespace and the dedicated tr_mutualfunds library, and S34 in both tfn and tr_13f. We use the dedicated libraries to avoid treating aliases as separate datasets. The consolidated s12 and s34 tables contain the holdings, investor attributes, security attributes, report dates, prices, and shares outstanding needed for initial extraction.

In [ ]:
SOURCE_TABLES = {
    "s12": {"library": "tr_mutualfunds", "table": "s12", "investor_id": "fundno"},
    "s34": {"library": "tr_13f", "table": "s34", "investor_id": "mgrno"},
}
source_table_frame = pd.DataFrame.from_dict(SOURCE_TABLES, orient="index").rename_axis("source").reset_index()
source_table_frame.to_parquet(catalog_directory / "selected_source_tables.parquet", index=False)
display(source_table_frame)

## 9. Query source date coverage

This low-cost database step requests only minimum and maximum file/report dates plus distinct date counts. It verifies whether each source covers the planned 2005Q1–2024Q4 window without downloading holdings rows. Results are stored locally.

In [ ]:
coverage_queries = {
    source: f"""
        select
            min(fdate) as min_fdate,
            max(fdate) as max_fdate,
            min(rdate) as min_rdate,
            max(rdate) as max_rdate,
            count(distinct fdate) as distinct_fdates,
            count(distinct rdate) as distinct_rdates
        from {config['library']}.{config['table']}
    """
    for source, config in SOURCE_TABLES.items()
}
coverage_frames = []
for source, query in coverage_queries.items():
    result = db.raw_sql(query, date_cols=["min_fdate", "max_fdate", "min_rdate", "max_rdate"])
    result.insert(0, "source", source)
    coverage_frames.append(result)
source_coverage = pd.concat(coverage_frames, ignore_index=True)
source_coverage.to_parquet(catalog_directory / "source_date_coverage.parquet", index=False)
display(source_coverage)

## 10. Download and save five-row source samples

This low-cost validation step downloads five observations from each consolidated holdings table. The samples let us confirm units, null patterns, identifiers, and date semantics before writing the pilot-quarter extraction. They are saved unchanged under data/raw/schema_samples.

In [ ]:
sample_directory = PROJECT_ROOT / "data/raw/schema_samples"
sample_directory.mkdir(parents=True, exist_ok=True)
source_samples = {}
for source, config in SOURCE_TABLES.items():
    query = f"select * from {config['library']}.{config['table']} limit 5"
    sample = db.raw_sql(query)
    source_samples[source] = sample
    sample.to_parquet(sample_directory / f"{source}_sample.parquet", index=False)
    print(f"{source.upper()} sample shape: {sample.shape}")
    display(sample)

## Next step: design and extract a pilot quarter

After validating coverage and field units, choose a quarter represented in both sources, download its S12 and S34 holdings into separate immutable Parquet files, and build source-specific diagnostics before attempting any overlap resolution.